# 11_02 Fine-tuning: can a small BERT beat TF-IDF on Kittiwake's tickets?

In Lab 03 you routed Kittiwake's support tickets to four departments with TF-IDF and logistic regression.
Here the same job goes to a BERT that has read Wikipedia. You add a four-way head, fine-tune, and measure
it against the old pipeline on exactly the same tickets.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-11-bert-and-what-came-after", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'transformers': 'transformers',
           'torch': 'torch',
           'sklearn': 'scikit-learn',
           'pandas': 'pandas',
           'numpy': 'numpy'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import os
import bertlab
from nlpcheck import ask, guess, reveal, check_11_02

os.makedirs("out", exist_ok=True)
results = {}
d = bertlab.tickets()
tr, te = bertlab.split()
print(len(tr), "training tickets,", len(te), "test tickets; departments", bertlab.DEPARTMENTS)

## 1. Recall

**r3.** In 11_01, why did "charge" get a different vector in each sentence?
(a) BERT computes each token's vector from the whole sentence around it, (b) it was looked up in a
dictionary of senses, (c) random noise

**r4.** Roughly how many weights does bert-mini have? (a) 110 million, (b) 11 million, (c) 1 million

In [ ]:
ask("r3", "")
ask("r4", "")

## 2. The number to beat

Lab 03's pipeline, trained on the 450 training tickets and scored on the 150 test tickets. Macro-F1
averages the four departments' F1 scores, so the small account team counts as much as billing.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000)).fit(d.text[tr], d.department[tr])
results["tfidf"] = bertlab.macro_f1(d.department[te], tfidf.predict(d.text[te]))
print("TF-IDF + logistic regression, macro-F1 on the 150:", results["tfidf"])

## 3. A pretrained BERT with an untrained head

`load_classifier(bertlab.TINY)` loads **bert-tiny** (2 layers, 128 numbers per token, about 4.4 million
weights) and puts a new layer on top: the vector of the `[CLS]` token in, four scores out, one per
department. The body is pretrained; the head is random numbers.

In [ ]:
model = bertlab.load_classifier(bertlab.TINY)
print(sum(p.numel() for p in model.parameters()), "weights,", sum(p.numel() for p in model.classifier.parameters()), "of them in the new head")
results["untrained"] = bertlab.macro_f1(d.department[te], bertlab.predict(model, d.text[te]))
print("untrained head, macro-F1:", results["untrained"])

About 0.09, below the 0.11 that answering "billing" to every ticket would score. Pretraining alone knows
nothing about departments.

## 4. Train only the head

The cheapest thing to try is to leave all the pretrained weights alone and train just the head: 516 numbers.
`fine_tune(..., freeze_bert=True)` does exactly that, for 5 passes over the 450 tickets. Predict the
macro-F1 on the 150 (a number between 0 and 1). It takes under a minute.

In [ ]:
guess("frozen_f1", None)   # a number, for example 0.8

In [ ]:
model = bertlab.load_classifier(bertlab.TINY)
secs = bertlab.fine_tune(model, d.text[tr], d.department[tr], epochs=5, lr=1e-3, freeze_bert=True)
results["frozen"] = bertlab.macro_f1(d.department[te], bertlab.predict(model, d.text[te]))
print(f"head only, {secs} s: macro-F1 {results['frozen']}")
reveal("frozen_f1", results["frozen"])

Far below TF-IDF. The `[CLS]` vector of a model pretrained to fill in blanks was never asked to summarise a
sentence's topic, so a single layer on top of it cannot pull the departments apart. The information is in
the network; it is not yet in that one vector.

## 5. Fine-tune everything

Now the recipe the BERT paper made standard: the same head, but **every** weight trains, the pretrained ones
with small steps. Five passes over 450 tickets on this machine's CPU takes about two minutes.

In [ ]:
model = bertlab.load_classifier(bertlab.TINY)
secs = bertlab.fine_tune(model, d.text[tr], d.department[tr], epochs=5, lr=3e-4)
results["tiny"] = bertlab.macro_f1(d.department[te], bertlab.predict(model, d.text[te]))
results["tiny_seconds"] = secs
print(f"bert-tiny fine-tuned in {secs} s: macro-F1 {results['tiny']} (TF-IDF: {results['tfidf']})")

From random to within a point of TF-IDF in five passes. That is what pretraining buys: a starting point from which
450 examples are enough.

## 6. Would a bigger BERT win?

bert-mini has two and a half times as many weights per layer and twice the layers. It takes several times
longer to fine-tune, so the session trained it once, in the background, when it started, with the same
split and five passes. Predict: does it beat TF-IDF on the 150 by more than two points, 0.02? (yes or no)

In [ ]:
guess("mini_beats_tfidf", None)   # "yes" or "no"

In [ ]:
import os
if not os.path.exists(bertlab.MINI_RESULT):
    print("The background run has not finished (or did not run), so it runs here now: up to about five minutes.")
mini = bertlab.mini_on_split()
results["mini"] = mini["macro_f1"]
print(f"bert-mini, {mini['weights']} weights, {mini['train_seconds']} s of training: macro-F1 {mini['macro_f1']}")
reveal("mini_beats_tfidf", "yes" if mini["macro_f1"] > results["tfidf"] + 0.02 else "no")

No. bert-mini, bert-tiny and TF-IDF land within about a point of each other, which on 150 tickets is one or
two tickets, and the reason is in
Lab 03: about one ticket in twenty was sent to the wrong department by Kittiwake's own triage team, which caps
every model near 0.92. Kittiwake's departments are also named by the words customers use (bill, signal,
phone), which is exactly what TF-IDF counts. Pretraining earns its keep where the words do not give the
answer away and labelled data is scarce; on this job the old pipeline is cheaper and as good, and a good
engineer says so.

## 7. Your turn: the ticket router, fine-tuned on everything

Fine-tune bert-tiny on **all 600** labelled tickets, the same way as section 5, and save it. The check loads
the saved folder and routes the 200 unlabelled tickets whose departments the checkpoint holds. Training
takes up to about three minutes.

In [ ]:
router = None   # YOUR CODE HERE: bertlab.load_classifier(bertlab.TINY)
# YOUR CODE HERE: fine-tune it on d.text and d.department with bertlab.fine_tune (epochs=5, lr=3e-4)

with open("out/11_02_results.json", "w") as f:
    json.dump(results, f, indent=1)
if router is None:
    print("Not yet: router is still None. Build it with bertlab.load_classifier and fine-tune it first.")
else:
    bertlab.save_classifier(router, "out/ticket_bert")
    check_11_02()

## 8. Exit ticket

In one or two sentences: why did training only the head score so much lower than fine-tuning everything?

*Your answer here.*

**x3.** Which three embeddings does BERT add together for every input token?
(a) word2vec, GloVe and TF-IDF, (b) token, segment and position, (c) query, key and value

In [ ]:
ask("x3", "")